In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [18]:
import os, json, random, copy
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2 as chi2_dist

# ── GPU check ──────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU.")

# ── Seeds ──────────────────────────────────────────────────
def set_seeds(seed=42):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seeds(42)
print("Seeds fixed at 42.")

Device: cuda
GPU:  Tesla T4
VRAM: 15.6 GB
Seeds fixed at 42.


In [19]:
# Clone your repo to get the saved baseline checkpoint
import subprocess

GITHUB_URL = "https://github.com/Ahmad-techs/fyp-food-classification.git" 

result = subprocess.run(["git", "clone", GITHUB_URL, "/kaggle/working/repo"],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)

# ── Paths ──────────────────────────────────────────────────
# Update FOOD11_ROOT to match your Kaggle input dataset path
# Check with: !ls /kaggle/input/
FOOD11_ROOT   = "/kaggle/input/datasets/trolukovich/food11-image-dataset"   
BASELINE_CKPT = "/kaggle/working/repo/models/best_model.pth"
OUTPUT_DIR    = "/kaggle/working/results"
CKPT_DIR      = "/kaggle/working/checkpoints"

for d in [OUTPUT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# Verify Food-11 structure
for split in ['training', 'validation', 'evaluation']:
    path = os.path.join(FOOD11_ROOT, split)
    if os.path.exists(path):
        classes = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
        total   = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
        print(f"  {split}/: {len(classes)} classes, ~{total} images ✓")
    else:
        print(f"  {split}/: NOT FOUND — check FOOD11_ROOT path")

fatal: destination path '/kaggle/working/repo' already exists and is not an empty directory.

  training/: 11 classes, ~9866 images ✓
  validation/: 11 classes, ~3430 images ✓
  evaluation/: 11 classes, ~3347 images ✓


In [20]:
# ── Coarse mapping: 11 Food-11 classes → 4 dietary groups ──
# Your existing grouping from coarse_mapping.py

CLASS_NAMES = [
    'Bread', 'Dairy product', 'Dessert', 'Egg', 'Fried food',
    'Meat', 'Noodles-Pasta', 'Rice', 'Seafood', 'Soup', 'Vegetable-Fruit'
]

# Group 0: Carbohydrate | 1: Protein | 2: Dessert/High-sugar | 3: Other/Light
COARSE_MAPPING = {
    'Bread': 0, 'Noodles-Pasta': 0, 'Rice': 0,
    'Meat': 1, 'Egg': 1, 'Seafood': 1, 'Dairy product': 1,
    'Dessert': 2,
    'Fried food': 3, 'Soup': 3, 'Vegetable-Fruit': 3
}

COARSE_NAMES = {
    0: 'Carbohydrate',
    1: 'Protein',
    2: 'Dessert / High-sugar',
    3: 'Other / Light'
}

def get_coarse_label(fine_name: str) -> int:
    return COARSE_MAPPING.get(fine_name, 3)

# Verify
print("Coarse mapping:")
for cls in CLASS_NAMES:
    gid = get_coarse_label(cls)
    print(f"  {cls:20s} → Group {gid} ({COARSE_NAMES[gid]})")

Coarse mapping:
  Bread                → Group 0 (Carbohydrate)
  Dairy product        → Group 1 (Protein)
  Dessert              → Group 2 (Dessert / High-sugar)
  Egg                  → Group 1 (Protein)
  Fried food           → Group 3 (Other / Light)
  Meat                 → Group 1 (Protein)
  Noodles-Pasta        → Group 0 (Carbohydrate)
  Rice                 → Group 0 (Carbohydrate)
  Seafood              → Group 1 (Protein)
  Soup                 → Group 3 (Other / Light)
  Vegetable-Fruit      → Group 3 (Other / Light)


In [21]:
# Handles Food-11 whether folders are named 'Bread'/'Meat'/... or '0'/'1'/...

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(split: str) -> transforms.Compose:
    if split == 'train':
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                   saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])
    else:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])

# Numeric folder → class name mapping (Food-11 numbered 0–10)
NUMERIC_TO_NAME = {
    '0': 'Bread', '1': 'Dairy product', '2': 'Dessert',
    '3': 'Egg',   '4': 'Fried food',    '5': 'Meat',
    '6': 'Noodles-Pasta', '7': 'Rice',  '8': 'Seafood',
    '9': 'Soup',  '10': 'Vegetable-Fruit'
}

class Food11Dataset(torch.utils.data.Dataset):
    def __init__(self, root_dir: str, split: str):
        self.transform = get_transforms(split)
        self.samples   = []   # (img_path, fine_label, coarse_label)

        split_dir = os.path.join(root_dir, split)
        folders   = sorted([d for d in os.listdir(split_dir)
                            if os.path.isdir(os.path.join(split_dir, d))])

        for folder in folders:
            # Resolve folder name → class name → fine/coarse ids
            if folder in NUMERIC_TO_NAME:
                class_name = NUMERIC_TO_NAME[folder]
                fine_id    = int(folder)
            elif folder in CLASS_NAMES:
                class_name = folder
                fine_id    = CLASS_NAMES.index(folder)
            else:
                print(f"  WARNING: unknown folder '{folder}' — skipping")
                continue

            coarse_id = get_coarse_label(class_name)
            folder_path = os.path.join(split_dir, folder)

            for fname in os.listdir(folder_path):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((
                        os.path.join(folder_path, fname),
                        fine_id, coarse_id
                    ))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, fine, coarse = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img), fine, coarse
        except Exception:
            return self.__getitem__((idx + 1) % len(self.samples))


# ── Create datasets and loaders ────────────────────────────
print("Building datasets...")
train_ds = Food11Dataset(FOOD11_ROOT, 'training')
val_ds   = Food11Dataset(FOOD11_ROOT, 'validation')
test_ds  = Food11Dataset(FOOD11_ROOT, 'evaluation')

print(f"  Train:      {len(train_ds)} images")
print(f"  Validation: {len(val_ds)} images")
print(f"  Test:       {len(test_ds)} images")

BS = 32
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

# Quick sanity check
imgs, fl, cl = next(iter(train_loader))
print(f"\nBatch shapes: imgs={imgs.shape} fine={fl.shape} coarse={cl.shape}")
print(f"Fine labels range:   [{fl.min().item()}, {fl.max().item()}]  (expected 0–10)")
print(f"Coarse labels range: [{cl.min().item()}, {cl.max().item()}]  (expected 0–3)")

Building datasets...
  Train:      9866 images
  Validation: 3430 images
  Test:       3347 images

Batch shapes: imgs=torch.Size([32, 3, 224, 224]) fine=torch.Size([32]) coarse=torch.Size([32])
Fine labels range:   [0, 10]  (expected 0–10)
Coarse labels range: [0, 3]  (expected 0–3)


In [22]:
class CoarseToFineNet(nn.Module):
    """
    EfficientNet-B0 backbone with dual heads.
    Returns (fine_out, coarse_out) — fine first, then coarse.
    """
    def __init__(self, num_fine=11, num_coarse=4, dropout=0.3):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features    = backbone.features
        self.avgpool     = backbone.avgpool
        self.dropout     = nn.Dropout(p=dropout)
        self.fine_head   = nn.Linear(1280, num_fine)
        self.coarse_head = nn.Linear(1280, num_coarse)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)   # (B, 1280)
        x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)
        # ← Always: (fine_out[B,11], coarse_out[B,4])

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False

    def unfreeze_top(self, n=3):
        for block in list(self.features.children())[-n:]:
            for p in block.parameters(): p.requires_grad = True

    def unfreeze_all(self):
        for p in self.features.parameters(): p.requires_grad = True


# ── Shape test ─────────────────────────────────────────────
with torch.no_grad():
    dummy       = torch.zeros(4, 3, 224, 224)
    m_test      = CoarseToFineNet()
    fine_t, coarse_t = m_test(dummy)
    assert fine_t.shape   == (4, 11), f"Expected (4,11) got {fine_t.shape}"
    assert coarse_t.shape == (4, 4),  f"Expected (4,4) got {coarse_t.shape}"
    total_p = sum(p.numel() for p in m_test.parameters())
    print(f"Model OK — fine: {fine_t.shape}  coarse: {coarse_t.shape}")
    print(f"Total parameters: {total_p:,}")
del m_test, dummy

Model OK — fine: torch.Size([4, 11])  coarse: torch.Size([4, 4])
Total parameters: 4,026,763


In [23]:
class CoarseToFineNet(nn.Module):
    """
    EfficientNet-B0 backbone with dual heads.
    Returns (fine_out, coarse_out) — fine first, then coarse.
    """
    def __init__(self, num_fine=11, num_coarse=4, dropout=0.3):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features    = backbone.features
        self.avgpool     = backbone.avgpool
        self.dropout     = nn.Dropout(p=dropout)
        self.fine_head   = nn.Linear(1280, num_fine)
        self.coarse_head = nn.Linear(1280, num_coarse)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)   # (B, 1280)
        x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)
        # ← Always: (fine_out[B,11], coarse_out[B,4])

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False

    def unfreeze_top(self, n=3):
        for block in list(self.features.children())[-n:]:
            for p in block.parameters(): p.requires_grad = True

    def unfreeze_all(self):
        for p in self.features.parameters(): p.requires_grad = True


# ── Shape test ─────────────────────────────────────────────
with torch.no_grad():
    dummy       = torch.zeros(4, 3, 224, 224)
    m_test      = CoarseToFineNet()
    fine_t, coarse_t = m_test(dummy)
    assert fine_t.shape   == (4, 11), f"Expected (4,11) got {fine_t.shape}"
    assert coarse_t.shape == (4, 4),  f"Expected (4,4) got {coarse_t.shape}"
    total_p = sum(p.numel() for p in m_test.parameters())
    print(f"Model OK — fine: {fine_t.shape}  coarse: {coarse_t.shape}")
    print(f"Total parameters: {total_p:,}")
del m_test, dummy

Model OK — fine: torch.Size([4, 11])  coarse: torch.Size([4, 4])
Total parameters: 4,026,763


In [24]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience; self.counter = 0; self.best = 0.0
    def step(self, acc):
        if acc > self.best: self.best = acc; self.counter = 0; return False
        self.counter += 1
        if self.counter >= self.patience:
            print(f"  Early stopping (patience={self.patience} reached)"); return True
        return False

@torch.no_grad()
def evaluate_val(model, loader):
    """Quick val evaluation: fine Top-1 and coarse Top-1."""
    model.eval()
    f_correct = c_correct = total = 0
    for imgs, fl, cl in loader:
        imgs, fl, cl = imgs.to(device), fl.to(device), cl.to(device)
        # ─── FIXED: fine_out first, coarse_out second ───
        fine_out, coarse_out = model(imgs)
        f_correct += (fine_out.argmax(1) == fl).sum().item()
        c_correct += (coarse_out.argmax(1) == cl).sum().item()
        total     += fl.size(0)
    return 100*f_correct/total, 100*c_correct/total


def run_phase(model, train_loader, val_loader, epochs, lr, save_path,
              lam, patience, history, phase_name):
    """One training phase. Saves checkpoint when val fine-accuracy improves."""
    set_seeds(42)
    crit = nn.CrossEntropyLoss()
    opt  = AdamW([p for p in model.parameters() if p.requires_grad],
                 lr=lr, weight_decay=1e-4)
    sch  = CosineAnnealingLR(opt, T_max=epochs)
    es   = EarlyStopping(patience=patience)
    best = history.get('best_fine_acc', 0.0)

    for ep in range(epochs):
        model.train(); epoch_loss = 0.0
        for imgs, fl, cl in train_loader:
            imgs, fl, cl = imgs.to(device), fl.to(device), cl.to(device)
            opt.zero_grad()
            # ─── FIXED: correct variable names ───────────
            fine_out, coarse_out = model(imgs)
            if lam == 0.0:
                # Baseline mode: only fine loss
                loss = crit(fine_out, fl)
            else:
                # Dual-head mode: weighted combined loss
                loss = lam * crit(coarse_out, cl) + (1 - lam) * crit(fine_out, fl)
            loss.backward(); opt.step()
            epoch_loss += loss.item()
        sch.step()

        f_acc, c_acc = evaluate_val(model, val_loader)
        n = len(train_loader)
        print(f"  {phase_name} Ep{ep+1:02d}/{epochs}  "
              f"loss={epoch_loss/n:.4f}  FineTop1={f_acc:.2f}%  CoarseTop1={c_acc:.2f}%")

        history.setdefault('val_fine_top1',   []).append(f_acc)
        history.setdefault('val_coarse_top1', []).append(c_acc)
        history.setdefault('train_loss',      []).append(epoch_loss/n)

        if f_acc > best:
            best = f_acc; history['best_fine_acc'] = best
            torch.save({'model_state_dict': model.state_dict(),
                        'best_fine_acc': best, 'lambda': lam,
                        'history': history}, save_path)
            print(f"  ✓ Checkpoint saved — Top-1 = {best:.2f}%")

        if es.step(f_acc): break
    return history


def train_experiment(lam: float, save_path: str) -> dict:
    """Full 3-phase training for one experimental condition."""
    print(f"\n{'='*60}")
    print(f"  λ = {lam}  |  save → {save_path}")
    print(f"{'='*60}")

    model = CoarseToFineNet(num_fine=11, num_coarse=4, dropout=0.3).to(device)
    history = {}

    print("\n── Phase 1: backbone FROZEN (5 epochs, lr=1e-3) ──────")
    model.freeze_backbone()
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable params: {n:,}")
    history = run_phase(model, train_loader, val_loader, 5, 1e-3,
                        save_path, lam, 5, history, "P1")

    print("\n── Phase 2: top 3 blocks UNFROZEN (15 epochs, lr=1e-4) ─")
    model.unfreeze_top(n=3)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable params: {n:,}")
    history = run_phase(model, train_loader, val_loader, 15, 1e-4,
                        save_path, lam, 5, history, "P2")

    print("\n── Phase 3: FULL backbone (5 epochs, lr=5e-5) ──────────")
    model.unfreeze_all()
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable params: {n:,}")
    history = run_phase(model, train_loader, val_loader, 5, 5e-5,
                        save_path, lam, 5, history, "P3")

    print(f"\n✓ Done. Best val Fine-Top-1: {history['best_fine_acc']:.2f}%")
    return history

In [25]:
# Load baseline checkpoint saved in your previous session
print("Loading baseline checkpoint...")

try:
    baseline_ckpt = torch.load(BASELINE_CKPT, map_location=device, weights_only=False)
    model_base    = CoarseToFineNet(num_fine=11, num_coarse=4).to(device)
    model_base.load_state_dict(baseline_ckpt['model_state_dict'])
    model_base.eval()

    f_acc, c_acc = evaluate_val(model_base, val_loader)
    print(f"Baseline checkpoint loaded ✓")
    print(f"  Val Fine-Top-1:   {f_acc:.2f}%")
    print(f"  Val Coarse-Top-1: {c_acc:.2f}%")
    print(f"  Saved best was:   {baseline_ckpt.get('best_fine_acc', 'N/A'):.2f}%")
    BASELINE_LOADED = True

except FileNotFoundError:
    print(f"Checkpoint not found at {BASELINE_CKPT}")
    print("Will re-train baseline now (λ=0.0)...")
    history_base = train_experiment(lam=0.0,
                                    save_path=f"{CKPT_DIR}/baseline_best.pth")
    baseline_ckpt = torch.load(f"{CKPT_DIR}/baseline_best.pth",
                                map_location=device, weights_only=False)
    model_base = CoarseToFineNet(num_fine=11, num_coarse=4).to(device)
    model_base.load_state_dict(baseline_ckpt['model_state_dict'])
    BASELINE_LOADED = True
    print("Baseline re-trained and loaded ✓")

Loading baseline checkpoint...
Baseline checkpoint loaded ✓
  Val Fine-Top-1:   90.85%
  Val Coarse-Top-1: 22.07%
  Saved best was:   91.25%


In [26]:
# Exp-1: equal coarse + fine supervision
print("Starting Exp-1: λ = 0.5")
history_e1 = train_experiment(
    lam       = 0.5,
    save_path = f"{CKPT_DIR}/exp1_lam05_best.pth"
)
with open(f"{OUTPUT_DIR}/exp1_history.json", 'w') as f:
    json.dump({k: v for k, v in history_e1.items() if isinstance(v, list)}, f)
print("Exp-1 history saved.")

Starting Exp-1: λ = 0.5

  λ = 0.5  |  save → /kaggle/working/checkpoints/exp1_lam05_best.pth

── Phase 1: backbone FROZEN (5 epochs, lr=1e-3) ──────
   Trainable params: 19,215
  P1 Ep01/5  loss=1.0618  FineTop1=75.36%  CoarseTop1=75.01%
  ✓ Checkpoint saved — Top-1 = 75.36%
  P1 Ep02/5  loss=0.7782  FineTop1=77.49%  CoarseTop1=77.08%
  ✓ Checkpoint saved — Top-1 = 77.49%
  P1 Ep03/5  loss=0.7229  FineTop1=78.60%  CoarseTop1=77.64%
  ✓ Checkpoint saved — Top-1 = 78.60%
  P1 Ep04/5  loss=0.6876  FineTop1=78.10%  CoarseTop1=78.05%
  P1 Ep05/5  loss=0.6735  FineTop1=78.02%  CoarseTop1=77.35%

── Phase 2: top 3 blocks UNFROZEN (15 epochs, lr=1e-4) ─
   Trainable params: 3,174,955
  P2 Ep01/15  loss=0.4931  FineTop1=86.59%  CoarseTop1=87.49%
  ✓ Checkpoint saved — Top-1 = 86.59%
  P2 Ep02/15  loss=0.2958  FineTop1=89.15%  CoarseTop1=89.33%
  ✓ Checkpoint saved — Top-1 = 89.15%
  P2 Ep03/15  loss=0.2045  FineTop1=89.65%  CoarseTop1=89.94%
  ✓ Checkpoint saved — Top-1 = 89.65%
  P2 Ep04/15  

In [27]:
print("Starting Exp-2: λ = 0.35")
history_e2 = train_experiment(
    lam       = 0.35,
    save_path = f"{CKPT_DIR}/exp2_lam035_best.pth"
)
with open(f"{OUTPUT_DIR}/exp2_history.json", 'w') as f:
    json.dump({k: v for k, v in history_e2.items() if isinstance(v, list)}, f)
print("Exp-2 history saved.")

Starting Exp-2: λ = 0.35

  λ = 0.35  |  save → /kaggle/working/checkpoints/exp2_lam035_best.pth

── Phase 1: backbone FROZEN (5 epochs, lr=1e-3) ──────
   Trainable params: 19,215
  P1 Ep01/5  loss=1.1174  FineTop1=75.04%  CoarseTop1=75.25%
  ✓ Checkpoint saved — Top-1 = 75.04%
  P1 Ep02/5  loss=0.7964  FineTop1=77.64%  CoarseTop1=76.47%
  ✓ Checkpoint saved — Top-1 = 77.64%
  P1 Ep03/5  loss=0.7354  FineTop1=78.78%  CoarseTop1=77.32%
  ✓ Checkpoint saved — Top-1 = 78.78%
  P1 Ep04/5  loss=0.6992  FineTop1=78.08%  CoarseTop1=77.61%
  P1 Ep05/5  loss=0.6822  FineTop1=78.13%  CoarseTop1=77.08%

── Phase 2: top 3 blocks UNFROZEN (15 epochs, lr=1e-4) ─
   Trainable params: 3,174,955
  P2 Ep01/15  loss=0.4939  FineTop1=86.97%  CoarseTop1=86.94%
  ✓ Checkpoint saved — Top-1 = 86.97%
  P2 Ep02/15  loss=0.2965  FineTop1=89.13%  CoarseTop1=89.15%
  ✓ Checkpoint saved — Top-1 = 89.13%
  P2 Ep03/15  loss=0.2036  FineTop1=89.88%  CoarseTop1=89.97%
  ✓ Checkpoint saved — Top-1 = 89.88%
  P2 Ep04/1

In [28]:
print("Starting Exp-3: λ = 0.2")
history_e3 = train_experiment(
    lam       = 0.2,
    save_path = f"{CKPT_DIR}/exp3_lam02_best.pth"
)
with open(f"{OUTPUT_DIR}/exp3_history.json", 'w') as f:
    json.dump({k: v for k, v in history_e3.items() if isinstance(v, list)}, f)
print("Exp-3 history saved.")

Starting Exp-3: λ = 0.2

  λ = 0.2  |  save → /kaggle/working/checkpoints/exp3_lam02_best.pth

── Phase 1: backbone FROZEN (5 epochs, lr=1e-3) ──────
   Trainable params: 19,215
  P1 Ep01/5  loss=1.1717  FineTop1=75.04%  CoarseTop1=75.25%
  ✓ Checkpoint saved — Top-1 = 75.04%
  P1 Ep02/5  loss=0.8159  FineTop1=77.64%  CoarseTop1=76.47%
  ✓ Checkpoint saved — Top-1 = 77.64%
  P1 Ep03/5  loss=0.7486  FineTop1=78.78%  CoarseTop1=77.32%
  ✓ Checkpoint saved — Top-1 = 78.78%
  P1 Ep04/5  loss=0.7099  FineTop1=78.08%  CoarseTop1=77.61%
  P1 Ep05/5  loss=0.6911  FineTop1=78.13%  CoarseTop1=77.08%

── Phase 2: top 3 blocks UNFROZEN (15 epochs, lr=1e-4) ─
   Trainable params: 3,174,955
  P2 Ep01/15  loss=0.4947  FineTop1=87.06%  CoarseTop1=86.18%
  ✓ Checkpoint saved — Top-1 = 87.06%
  P2 Ep02/15  loss=0.2962  FineTop1=89.04%  CoarseTop1=88.78%
  ✓ Checkpoint saved — Top-1 = 89.04%
  P2 Ep03/15  loss=0.2015  FineTop1=89.74%  CoarseTop1=89.59%
  ✓ Checkpoint saved — Top-1 = 89.74%
  P2 Ep04/15  

In [29]:
#   This cell runs ONCE for dissertation results.

@torch.no_grad()
def full_test_eval(model, loader, label):
    """Complete evaluation: Top-1, Top-5, F1, per-image correctness for McNemar."""
    model.eval()
    all_ft, all_fp       = [], []
    all_ct, all_cp       = [], []
    all_fine_probs       = []

    for imgs, fl, cl in loader:
        imgs = imgs.to(device)
        fine_out, coarse_out = model(imgs)
        probs = torch.softmax(fine_out, dim=1).cpu().numpy()
        all_fine_probs.extend(probs)
        all_fp.extend(fine_out.argmax(1).cpu().numpy())
        all_ft.extend(fl.numpy())
        all_cp.extend(coarse_out.argmax(1).cpu().numpy())
        all_ct.extend(cl.numpy())

    all_ft = np.array(all_ft); all_fp = np.array(all_fp)
    all_ct = np.array(all_ct); all_cp = np.array(all_cp)
    all_fine_probs = np.array(all_fine_probs)

    # Fine metrics
    fine_top1 = accuracy_score(all_ft, all_fp) * 100
    fine_f1   = f1_score(all_ft, all_fp, average='macro', zero_division=0) * 100
    # Top-5 (since only 11 classes, top-5 is very easy but still report it)
    top5_correct = sum(
        all_ft[i] in np.argsort(all_fine_probs[i])[-5:]
        for i in range(len(all_ft))
    )
    fine_top5 = 100 * top5_correct / len(all_ft)

    # Coarse metric
    coarse_top1 = accuracy_score(all_ct, all_cp) * 100

    # Per-image correctness (needed for McNemar test)
    fine_correct = (all_fp == all_ft).astype(int)

    print(f"\n{'='*55}")
    print(f"  {label}  [TEST SET]")
    print(f"{'='*55}")
    print(f"  Fine Top-1:   {fine_top1:.2f}%")
    print(f"  Fine Top-5:   {fine_top5:.2f}%")
    print(f"  Fine F1:      {fine_f1:.2f}%")
    print(f"  Coarse Top-1: {coarse_top1:.2f}%")

    # Per-dietary-group fine accuracy
    print(f"\n  Fine accuracy per dietary group:")
    for gid, gname in COARSE_NAMES.items():
        mask = all_ct == gid
        if mask.sum() > 0:
            acc = accuracy_score(all_ft[mask], all_fp[mask]) * 100
            print(f"    {gname:25s}: {acc:.1f}%  ({mask.sum()} images)")

    return {
        'condition':    label,
        'fine_top1':    fine_top1,
        'fine_top5':    fine_top5,
        'fine_f1':      fine_f1,
        'coarse_top1':  coarse_top1,
        'fine_correct': fine_correct,
        'all_ft': all_ft, 'all_fp': all_fp,
        'all_ct': all_ct, 'all_cp': all_cp,
    }


# Load all checkpoints
experiments = [
    ('Baseline',    BASELINE_CKPT if os.path.exists(BASELINE_CKPT)
                    else f"{CKPT_DIR}/baseline_best.pth"),
    ('Dual λ=0.5',  f"{CKPT_DIR}/exp1_lam05_best.pth"),
    ('Dual λ=0.35', f"{CKPT_DIR}/exp2_lam035_best.pth"),
    ('Dual λ=0.2',  f"{CKPT_DIR}/exp3_lam02_best.pth"),
]

all_test_results = []
for label, ckpt_path in experiments:
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = CoarseToFineNet(num_fine=11, num_coarse=4).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    result = full_test_eval(model, test_loader, label)
    all_test_results.append(result)

print("\n\n" + "="*60)
print("TEST SET SUMMARY")
print("="*60)
for r in all_test_results:
    print(f"  {r['condition']:15s}  "
          f"Top1={r['fine_top1']:.2f}%  "
          f"Top5={r['fine_top5']:.2f}%  "
          f"F1={r['fine_f1']:.2f}%  "
          f"Coarse={r['coarse_top1']:.2f}%")


  Baseline  [TEST SET]
  Fine Top-1:   92.23%
  Fine Top-5:   99.73%
  Fine F1:      92.58%
  Coarse Top-1: 23.24%

  Fine accuracy per dietary group:
    Carbohydrate             : 92.6%  (611 images)
    Protein                  : 90.6%  (1218 images)
    Dessert / High-sugar     : 89.8%  (500 images)
    Other / Light            : 95.2%  (1018 images)

  Dual λ=0.5  [TEST SET]
  Fine Top-1:   93.25%
  Fine Top-5:   99.67%
  Fine F1:      93.58%
  Coarse Top-1: 93.07%

  Fine accuracy per dietary group:
    Carbohydrate             : 93.6%  (611 images)
    Protein                  : 92.2%  (1218 images)
    Dessert / High-sugar     : 90.4%  (500 images)
    Other / Light            : 95.7%  (1018 images)

  Dual λ=0.35  [TEST SET]
  Fine Top-1:   93.43%
  Fine Top-5:   99.70%
  Fine F1:      93.69%
  Coarse Top-1: 93.25%

  Fine accuracy per dietary group:
    Carbohydrate             : 94.3%  (611 images)
    Protein                  : 92.2%  (1218 images)
    Dessert / High-sugar

In [30]:
def mcnemar_test(result_a, result_b):
    ca, cb = result_a['fine_correct'], result_b['fine_correct']
    assert len(ca) == len(cb)
    n01  = int(np.sum((ca==0) & (cb==1)))  # B correct, A wrong
    n10  = int(np.sum((ca==1) & (cb==0)))  # A correct, B wrong
    chi2 = (abs(n01-n10)-1)**2/(n01+n10) if (n01+n10)>0 else 0.0
    p    = 1 - chi2_dist.cdf(chi2, df=1)
    sig  = "✓ SIGNIFICANT (p<0.05)" if p < 0.05 else "✗ NOT significant"
    print(f"  McNemar: {result_a['condition']} vs {result_b['condition']}")
    print(f"    n01={n01}  n10={n10}  χ²={chi2:.4f}  p={p:.4f}  → {sig}")
    return chi2, p

def bootstrap_ci(correct, n=1000):
    means = [np.random.choice(correct, len(correct), replace=True).mean()*100
             for _ in range(n)]
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

baseline_r = all_test_results[0]
print("Statistical Analysis (Test Set)\n")

print("McNemar tests vs Baseline:")
for r in all_test_results[1:]:
    mcnemar_test(baseline_r, r)

print("\n95% Bootstrap Confidence Intervals (1000 resamples):")
for r in all_test_results:
    lo, hi = bootstrap_ci(r['fine_correct'])
    print(f"  {r['condition']:15s}: {r['fine_top1']:.2f}%  CI=[{lo:.2f}%, {hi:.2f}%]")

# Find best dual-head
dual_results = all_test_results[1:]
best = max(dual_results, key=lambda r: r['fine_top1'])
diff = best['fine_top1'] - baseline_r['fine_top1']
print(f"\nBest dual-head: {best['condition']}  "
      f"({best['fine_top1']:.2f}% vs baseline {baseline_r['fine_top1']:.2f}%  "
      f"Δ={diff:+.2f}pp)")

Statistical Analysis (Test Set)

McNemar tests vs Baseline:
  McNemar: Baseline vs Dual λ=0.5
    n01=89  n10=55  χ²=7.5625  p=0.0060  → ✓ SIGNIFICANT (p<0.05)
  McNemar: Baseline vs Dual λ=0.35
    n01=93  n10=53  χ²=10.4178  p=0.0012  → ✓ SIGNIFICANT (p<0.05)
  McNemar: Baseline vs Dual λ=0.2
    n01=89  n10=56  χ²=7.0621  p=0.0079  → ✓ SIGNIFICANT (p<0.05)

95% Bootstrap Confidence Intervals (1000 resamples):
  Baseline       : 92.23%  CI=[91.37%, 93.16%]
  Dual λ=0.5     : 93.25%  CI=[92.41%, 94.14%]
  Dual λ=0.35    : 93.43%  CI=[92.59%, 94.26%]
  Dual λ=0.2     : 93.22%  CI=[92.38%, 94.05%]

Best dual-head: Dual λ=0.35  (93.43% vs baseline 92.23%  Δ=+1.20pp)


In [31]:
conditions = [r['condition'] for r in all_test_results]
fine_top1  = [r['fine_top1']  for r in all_test_results]
fine_top5  = [r['fine_top5']  for r in all_test_results]
coarse_acc = [r['coarse_top1'] for r in all_test_results]
colours    = ['#c0392b', '#27ae60', '#2980b9', '#8e44ad']
x          = np.arange(len(conditions))

# ── Chart 1: 3-panel comparison bar chart ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Coarse-to-Fine Food-11 Classification — Test Set Results',
             fontsize=13, fontweight='bold')
for ax, vals, title in [
    (axes[0], fine_top1,  'Fine Top-1 Accuracy (%)'),
    (axes[1], fine_top5,  'Fine Top-5 Accuracy (%)'),
    (axes[2], coarse_acc, 'Coarse Top-1 Accuracy (%)')
]:
    bars = ax.bar(x, vals, color=colours, edgecolor='white')
    ax.set_title(title, fontsize=11)
    ax.set_xticks(x); ax.set_xticklabels(conditions, rotation=15, ha='right', fontsize=9)
    ax.set_ylim(0, 100)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{v:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
chart1 = f"{OUTPUT_DIR}/test_comparison_chart.png"
plt.savefig(chart1, dpi=300, bbox_inches='tight'); plt.close()
print(f"Chart 1 saved: {chart1}")

# ── Chart 2: Coarse confusion matrix for best dual-head ───
best_r = max(all_test_results[1:], key=lambda r: r['fine_top1'])
cm     = confusion_matrix(best_r['all_ct'], best_r['all_cp'])
fig, ax = plt.subplots(figsize=(7, 6))
gnames  = [COARSE_NAMES[i] for i in range(4)]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=gnames, yticklabels=gnames, ax=ax)
ax.set_xlabel('Predicted Group', fontsize=11)
ax.set_ylabel('True Group', fontsize=11)
ax.set_title(f'Coarse Confusion Matrix — {best_r["condition"]}', fontsize=11)
plt.xticks(rotation=20, ha='right', fontsize=9); plt.tight_layout()
chart2 = f"{OUTPUT_DIR}/{best_r['condition'].replace(' ','_')}_coarse_cm.png"
plt.savefig(chart2, dpi=300, bbox_inches='tight'); plt.close()
print(f"Chart 2 saved: {chart2}")

# ── Chart 3: 11×11 fine confusion matrix for best model ───
fine_cm  = confusion_matrix(best_r['all_ft'], best_r['all_fp'])
fig, ax  = plt.subplots(figsize=(12, 10))
sns.heatmap(fine_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted Class', fontsize=10); ax.set_ylabel('True Class', fontsize=10)
ax.set_title(f'Fine Confusion Matrix — {best_r["condition"]}', fontsize=11)
plt.xticks(rotation=35, ha='right', fontsize=8)
plt.tight_layout()
chart3 = f"{OUTPUT_DIR}/{best_r['condition'].replace(' ','_')}_fine_cm.png"
plt.savefig(chart3, dpi=300, bbox_inches='tight'); plt.close()
print(f"Chart 3 saved: {chart3}")

# ── Chart 4: Training curves ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labels_h  = ['Dual λ=0.5', 'Dual λ=0.35', 'Dual λ=0.2']
histories = [history_e1, history_e2, history_e3]
cols      = ['#27ae60', '#2980b9', '#8e44ad']
for h, lbl, c in zip(histories, labels_h, cols):
    axes[0].plot(h.get('train_loss',    []), label=lbl, color=c)
    axes[1].plot(h.get('val_fine_top1', []), label=lbl, color=c)
for ax, title, ylabel in [
    (axes[0], 'Training Loss (Dual-Head Experiments)',    'Loss'),
    (axes[1], 'Val Fine Top-1 Accuracy (%) — Dual-Head', 'Accuracy (%)')
]:
    ax.set_title(title, fontsize=11); ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel); ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
chart4 = f"{OUTPUT_DIR}/dual_head_training_curves.png"
plt.savefig(chart4, dpi=300, bbox_inches='tight'); plt.close()
print(f"Chart 4 saved: {chart4}")
print("\nAll charts saved at 300 dpi — ready to insert into dissertation.")

Chart 1 saved: /kaggle/working/results/test_comparison_chart.png
Chart 2 saved: /kaggle/working/results/Dual_λ=0.35_coarse_cm.png
Chart 3 saved: /kaggle/working/results/Dual_λ=0.35_fine_cm.png
Chart 4 saved: /kaggle/working/results/dual_head_training_curves.png

All charts saved at 300 dpi — ready to insert into dissertation.


In [32]:
import csv

rows = []
for r in all_test_results:
    lo, hi = bootstrap_ci(r['fine_correct'])
    rows.append({
        'Condition':    r['condition'],
        'Fine_Top1':    round(r['fine_top1'],  2),
        'Fine_Top5':    round(r['fine_top5'],  2),
        'Fine_F1':      round(r['fine_f1'],    2),
        'Coarse_Top1':  round(r['coarse_top1'],2),
        'CI_Low':       round(lo,  2),
        'CI_High':      round(hi,  2),
    })

csv_path = f"{OUTPUT_DIR}/TEST_RESULTS_FINAL.csv"
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader(); writer.writerows(rows)

print("Final results saved:", csv_path)
print("\nFinal results table:")
print(f"{'Condition':15s} {'Top-1':>7} {'Top-5':>7} {'F1':>7} {'Coarse':>8} {'CI':>18}")
print("-" * 65)
for row in rows:
    print(f"{row['Condition']:15s} "
          f"{row['Fine_Top1']:>7.2f} "
          f"{row['Fine_Top5']:>7.2f} "
          f"{row['Fine_F1']:>7.2f} "
          f"{row['Coarse_Top1']:>8.2f} "
          f"[{row['CI_Low']:.2f}%, {row['CI_High']:.2f}%]")

print("\n  TEST SET IS NOW CLOSED. Do not re-run Cell 15.")
print("Download all files from /kaggle/working/results/ before closing this session.")

Final results saved: /kaggle/working/results/TEST_RESULTS_FINAL.csv

Final results table:
Condition         Top-1   Top-5      F1   Coarse                 CI
-----------------------------------------------------------------
Baseline          92.23   99.73   92.58    23.24 [91.34%, 93.10%]
Dual λ=0.5        93.25   99.67   93.58    93.07 [92.38%, 94.08%]
Dual λ=0.35       93.43   99.70   93.69    93.25 [92.56%, 94.29%]
Dual λ=0.2        93.22   99.70   93.53    92.98 [92.44%, 93.99%]

  TEST SET IS NOW CLOSED. Do not re-run Cell 15.
Download all files from /kaggle/working/results/ before closing this session.


In [33]:
import os

print("Files in /kaggle/working/:")
for file in os.listdir('/kaggle/working'):
    print(file)

Files in /kaggle/working/:
.virtual_documents
.ipynb_checkpoints
repo
results
checkpoints
exp3_lam02.pth


In [34]:
import os

print("Searching for .pth files in /kaggle/working/...")
found_files = []

for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file.endswith('.pth'):
            path = os.path.join(root, file)
            found_files.append(path)
            print(f"Found: {path}")

if not found_files:
    print("\nNo .pth files found anywhere in the /kaggle/working directory.")
    print("If you see this, the training code did not successfully save the checkpoints.")

Searching for .pth files in /kaggle/working/...
Found: /kaggle/working/exp3_lam02.pth
Found: /kaggle/working/repo/models/best_model.pth
Found: /kaggle/working/checkpoints/exp2_lam035_best.pth
Found: /kaggle/working/checkpoints/exp3_lam02_best.pth
Found: /kaggle/working/checkpoints/exp1_lam05_best.pth


In [36]:
import os
ckpt_dir = "/kaggle/working/checkpoints"
if os.path.exists(ckpt_dir):
    for f in os.listdir(ckpt_dir):
        size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
        print(f"{f}  ({size_mb:.1f} MB)  → download this now")
else:
    print("Session reset — checkpoints gone. Re-run Cells 8-10 (Day 1 below).")

exp2_lam035_best.pth  (16.4 MB)  → download this now
exp3_lam02_best.pth  (16.4 MB)  → download this now
exp1_lam05_best.pth  (16.4 MB)  → download this now


In [37]:
import shutil
import os

# Zip the checkpoints folder
shutil.make_archive('/kaggle/working/all_models', 'zip', '/kaggle/working/checkpoints')

print("Zipping complete. Look for 'all_models.zip' in your sidebar.")

Zipping complete. Look for 'all_models.zip' in your sidebar.


In [38]:
from IPython.display import FileLink

# This will generate a direct download link
display(FileLink('all_models.zip'))

/kaggle/working/all_models.zip